# AnchorKV: physical requantization on a real Qwen KV cache

This standalone Colab notebook tests the new backend architecture on **Qwen/Qwen3-0.6B**. It does not need repository access. The experiment captures the model's real prompt KV cache, stores token segments as FP16, groupwise INT8, or packed INT4, reconstructs a Hugging Face `DynamicCache`, and measures controlled teacher-forced drift plus greedy generation drift.

The declarative policy uses `<anchor segments="...">` and `<archive segments="...">` directives. Anchored segments stay FP16 while archived context becomes INT4. This is a reference correctness path: the packed-byte count is physical and exact, but standard Hugging Face attention requires a dense materialized cache, so **CUDA peak memory is not expected to fall yet**. A fused/paged quantized-attention kernel is the later systems step.

Runtime: select **Runtime → Change runtime type → T4 GPU**, then run all cells. Expected runtime is a few minutes after the model download.

In [ ]:
%pip install -q "transformers>=5.0,<6" "huggingface_hub>=0.34" matplotlib pandas

In [ ]:
import gc
import json
import math
import platform
import re
import time
from dataclasses import dataclass
from enum import Enum
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

MODEL_ID = 'Qwen/Qwen3-0.6B'
MAX_PROMPT_TOKENS = 640
MAX_NEW_TOKENS = 32
TEACHER_FORCED_STEPS = 24
GROUP_SIZE = 64
SEED = 7
OUTPUT_DIR = Path('/content/anchorkv-requantization')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. In Colab select a T4 GPU runtime.')
DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
if 'T4' not in GPU_NAME:
    print(f'Warning: designed for a T4; detected {GPU_NAME!r}.')
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print({'gpu': GPU_NAME, 'torch': torch.__version__, 'transformers': transformers.__version__})

## Embedded physical backend

INT4 values are truly packed two per byte. Every scale tensor is included in the resident-byte count. Quantization is performed per group across each layer/segment tensor.

In [ ]:
class CacheMode(str, Enum):
    FP16 = 'fp16'
    INT8 = 'int8'
    INT4 = 'int4'


@dataclass(frozen=True)
class PackedTensor:
    mode: CacheMode
    shape: tuple
    original_dtype: torch.dtype
    payload: torch.Tensor
    scales: torch.Tensor | None
    group_size: int
    original_numel: int

    @property
    def stored_bytes(self):
        tensors = (self.payload, self.scales)
        return sum(t.numel() * t.element_size() for t in tensors if t is not None)

    def dequantize(self, *, dtype=torch.float16, device=DEVICE):
        payload = self.payload.to(device)
        if self.mode is CacheMode.FP16:
            return payload.to(dtype=dtype).reshape(self.shape)
        scales = self.scales.to(device=device, dtype=torch.float32)
        if self.mode is CacheMode.INT8:
            quantized = payload.to(torch.float32)
        else:
            low = torch.bitwise_and(payload, 0x0F)
            high = torch.bitwise_right_shift(payload, 4)
            quantized = torch.stack((low, high), dim=-1).reshape(-1).to(torch.int16)
            quantized = torch.where(quantized >= 8, quantized - 16, quantized).float()
        padded_numel = scales.numel() * self.group_size
        values = quantized[:padded_numel].reshape(-1, self.group_size)
        values = values * scales.reshape(-1, 1)
        return values.reshape(-1)[:self.original_numel].reshape(self.shape).to(dtype)


def quantize_tensor(tensor, mode, group_size=GROUP_SIZE):
    source = tensor.detach().contiguous().cpu()
    shape = tuple(source.shape)
    original_numel = source.numel()
    if mode is CacheMode.FP16:
        return PackedTensor(mode, shape, source.dtype, source.half(), None, group_size, original_numel)
    flat = source.float().reshape(-1)
    padded_numel = math.ceil(original_numel / group_size) * group_size
    if padded_numel > original_numel:
        flat = F.pad(flat, (0, padded_numel - original_numel))
    groups = flat.reshape(-1, group_size)
    qmax = 127 if mode is CacheMode.INT8 else 7
    maxima = groups.abs().amax(dim=1)
    scales = torch.where(maxima > 0, maxima / qmax, torch.ones_like(maxima)).half()
    quantized = torch.round(groups / scales.float()[:, None]).clamp(-qmax, qmax).to(torch.int8)
    if mode is CacheMode.INT8:
        payload = quantized.reshape(-1).contiguous()
    else:
        encoded = torch.bitwise_and(quantized.to(torch.int16), 0x0F).to(torch.uint8).reshape(-1)
        payload = torch.bitwise_or(encoded[0::2], torch.bitwise_left_shift(encoded[1::2], 4)).contiguous()
    return PackedTensor(mode, shape, source.dtype, payload, scales, group_size, original_numel)


@dataclass(frozen=True)
class Segment:
    segment_id: int
    name: str
    start: int
    end: int


@dataclass
class PackedCache:
    layers: list
    segments: list
    modes: dict

    @property
    def resident_bytes(self):
        return sum(item.stored_bytes for layer in self.layers for pair in layer for item in pair)

    @property
    def fp16_bytes(self):
        return sum(item.original_numel * 2 for layer in self.layers for pair in layer for item in pair)

    def report(self):
        by_mode = {}
        for segment in self.segments:
            mode = self.modes[segment.segment_id].value
            by_mode[mode] = by_mode.get(mode, 0) + segment.end - segment.start
        return {
            'resident_bytes': self.resident_bytes,
            'fp16_bytes': self.fp16_bytes,
            'compression_ratio': self.fp16_bytes / self.resident_bytes,
            'tokens_by_mode': by_mode,
        }

    def materialize(self, model):
        cache_data = []
        for layer in self.layers:
            keys = torch.cat([pair[0].dequantize(dtype=model.dtype) for pair in layer], dim=-2)
            values = torch.cat([pair[1].dequantize(dtype=model.dtype) for pair in layer], dim=-2)
            cache_data.append((keys, values))
        # Transformers 5 accepts DDP-style layer tuples. Keep fallbacks for late 4.x.
        for constructor in (
            lambda: DynamicCache(cache_data),
            lambda: DynamicCache(ddp_cache_data=cache_data),
        ):
            try:
                return constructor()
            except TypeError:
                pass
        cache = DynamicCache(config=model.config)
        for layer_idx, (keys, values) in enumerate(cache_data):
            cache.update(keys, values, layer_idx)
        return cache


def cache_layer_tuples(cache):
    if hasattr(cache, 'layers'):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, 'key_cache') and hasattr(cache, 'value_cache'):
        return list(zip(cache.key_cache, cache.value_cache))
    if hasattr(cache, 'to_legacy_cache'):
        return list(cache.to_legacy_cache())
    return list(cache)


def snapshot_cache_to_cpu(cache):
    return [(key.detach().cpu(), value.detach().cpu()) for key, value in cache_layer_tuples(cache)]


def pack_cache(source_layers, segments, modes):
    packed_layers = []
    for key, value in source_layers:
        layer = []
        for segment in segments:
            mode = modes[segment.segment_id]
            layer.append((
                quantize_tensor(key[..., segment.start:segment.end, :], mode),
                quantize_tensor(value[..., segment.start:segment.end, :], mode),
            ))
        packed_layers.append(layer)
    return PackedCache(packed_layers, segments, modes)

## Declarative thought-anchor policy

The prompt is assembled from separately tokenized segments, making the KV boundaries exact. The relevant evidence is deliberately placed in the middle of a longer context. The declarative policy anchors the instruction, relevant evidence, and query; it archives distractors to INT4.

In [ ]:
SEGMENT_ATTR = re.compile(r"(?:segments|magic_chunks)\s*=\s*['\"]([0-9,\s]+)['\"]", re.I)


def parse_declarative_policy(text, segment_ids):
    modes = {segment_id: CacheMode.FP16 for segment_id in segment_ids}
    anchored = set()
    for raw_tag in re.findall(r'<([^>]+)>', text):
        tag = raw_tag.strip()
        match = SEGMENT_ATTR.search(tag)
        ids = [] if match is None else [int(value.strip()) for value in match.group(1).split(',')]
        unknown = set(ids) - set(segment_ids)
        if unknown:
            raise ValueError(f'Unknown segment IDs: {sorted(unknown)}')
        if tag.lower().startswith('anchor'):
            anchored.update(ids)
        elif tag.lower().startswith('archive'):
            for segment_id in ids:
                if segment_id in anchored:
                    raise ValueError(f'Cannot archive anchored segment {segment_id}')
                modes[segment_id] = CacheMode.INT4
    return modes


SEGMENT_TEXT = [
    ('instruction', 'You are a careful retrieval assistant. Answer the final question with only the requested project code.\n'),
    ('distractor-1', 'Archive memo one discusses blueprints, maintenance windows, and old inventory labels. None of its identifiers answer the final question. ' * 4 + '\n'),
    ('distractor-2', 'Archive memo two contains weather observations, shipping routes, and calibration notes. Treat its numbers as irrelevant noise: 1842, 5601, 9920. ' * 4 + '\n'),
    ('distractor-3', 'Archive memo three summarizes meeting rooms, stationery orders, and obsolete account references. It is unrelated to Project Zephyr. ' * 4 + '\n'),
    ('zephyr-evidence', 'AUTHORITATIVE RECORD: The four-digit access code assigned to Project Zephyr is 7319. This record supersedes every other memo.\n'),
    ('distractor-4', 'Archive memo four lists sample codes 4428, 1170, and 8834 for retired projects. These are decoys and must not be returned. ' * 4 + '\n'),
    ('distractor-5', 'Archive memo five describes cafeteria schedules and equipment reservations. It provides no valid Project Zephyr credential. ' * 4 + '\n'),
    ('query', 'Question: What is the four-digit access code for Project Zephyr? Answer with the four digits only.\nAnswer:'),
]
TARGET_SEGMENT_ID = 4

declarative_program = (
    '<anchor segments="0,4,7">'
    '<archive segments="1,2,3,5,6">'
    '<focus segments="4">'
)
print('Declarative program:', declarative_program)

In [ ]:
revision = model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=revision,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(DEVICE).eval()

token_parts = [tokenizer(text, add_special_tokens=False).input_ids for _, text in SEGMENT_TEXT]
all_ids = []
segments = []
for segment_id, ((name, _), ids) in enumerate(zip(SEGMENT_TEXT, token_parts)):
    start = len(all_ids)
    all_ids.extend(ids)
    segments.append(Segment(segment_id, name, start, len(all_ids)))
if len(all_ids) > MAX_PROMPT_TOKENS:
    raise RuntimeError(f'Prompt has {len(all_ids)} tokens; cap is {MAX_PROMPT_TOKENS}.')
input_ids = torch.tensor([all_ids], device=DEVICE, dtype=torch.long)
attention_mask = torch.ones_like(input_ids)
print(f'Model revision: {revision}')
print(f'Prompt tokens: {input_ids.shape[1]}')
pd.DataFrame([vars(segment) | {'tokens': segment.end - segment.start} for segment in segments])

## Capture the real model cache and pack each policy

The source cache is copied to CPU once, then the GPU copy is deleted. This keeps the evaluation bounded on a T4 and avoids counting a hidden second FP16 source cache in later CUDA measurements.

In [ ]:
with torch.inference_mode():
    prefill = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
first_logits = prefill.logits[:, -1, :].detach().cpu()
source_layers = snapshot_cache_to_cpu(prefill.past_key_values)
del prefill
gc.collect()
torch.cuda.empty_cache()

segment_ids = [segment.segment_id for segment in segments]
declarative_modes = parse_declarative_policy(declarative_program, segment_ids)
policies = {
    'fp16': {segment_id: CacheMode.FP16 for segment_id in segment_ids},
    'uniform_int8': {segment_id: CacheMode.INT8 for segment_id in segment_ids},
    'uniform_int4': {segment_id: CacheMode.INT4 for segment_id in segment_ids},
    'declarative_int4': declarative_modes,
}

packed_caches = {}
for policy_name, modes in policies.items():
    started = time.perf_counter()
    packed_caches[policy_name] = pack_cache(source_layers, segments, modes)
    report = packed_caches[policy_name].report()
    print(policy_name, report, f'pack_seconds={time.perf_counter() - started:.2f}')

storage_table = pd.DataFrame([
    {'policy': name, **packed.report()} for name, packed in packed_caches.items()
])
storage_table['resident_mib'] = storage_table.resident_bytes / 2**20
storage_table[['policy', 'resident_mib', 'compression_ratio', 'tokens_by_mode']]

## Controlled teacher-forced evaluation

First create a deterministic FP16 continuation. Then replay those same tokens under every cache policy. Because every policy sees identical token history, KL divergence and top-1 agreement isolate the effect of prompt-cache precision instead of compounding differences from divergent text.

In [ ]:
def run_decode(packed, forced_tokens=None, max_new_tokens=MAX_NEW_TOKENS):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    cache = packed.materialize(model)
    next_logits = first_logits.to(DEVICE)
    generated = []
    step_logits = []
    started = time.perf_counter()
    steps = len(forced_tokens) if forced_tokens is not None else max_new_tokens
    with torch.inference_mode():
        for step in range(steps):
            if forced_tokens is None:
                token = next_logits.argmax(dim=-1)
                generated.append(int(token.item()))
            else:
                token = torch.tensor([forced_tokens[step]], device=DEVICE)
            cache_length = int(cache.get_seq_length())
            mask = torch.ones((1, cache_length + 1), dtype=torch.long, device=DEVICE)
            output = model(
                input_ids=token.reshape(1, 1),
                attention_mask=mask,
                past_key_values=cache,
                cache_position=torch.tensor([cache_length], device=DEVICE),
                use_cache=True,
            )
            cache = output.past_key_values
            next_logits = output.logits[:, -1, :]
            step_logits.append(next_logits.detach().float().cpu())
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    peak_gpu_gib = torch.cuda.max_memory_allocated() / 2**30
    del cache, next_logits
    gc.collect()
    torch.cuda.empty_cache()
    return {
        'tokens': generated,
        'logits': torch.cat(step_logits, dim=0),
        'seconds': elapsed,
        'peak_gpu_gib': peak_gpu_gib,
    }


fp16_free = run_decode(packed_caches['fp16'])
baseline_tokens = fp16_free['tokens']
baseline_text = tokenizer.decode(baseline_tokens, skip_special_tokens=True)
teacher_tokens = baseline_tokens[:TEACHER_FORCED_STEPS]
fp16_teacher = run_decode(packed_caches['fp16'], forced_tokens=teacher_tokens)
print('FP16 greedy continuation:', repr(baseline_text))

In [ ]:
def causal_kl(reference_logits, candidate_logits):
    ref_log_probs = F.log_softmax(reference_logits.float(), dim=-1)
    candidate_log_probs = F.log_softmax(candidate_logits.float(), dim=-1)
    return (ref_log_probs.exp() * (ref_log_probs - candidate_log_probs)).sum(dim=-1)


results = []
free_outputs = {'fp16': fp16_free}
for policy_name, packed in packed_caches.items():
    if policy_name == 'fp16':
        teacher = fp16_teacher
        free = fp16_free
    else:
        teacher = run_decode(packed, forced_tokens=teacher_tokens)
        free = run_decode(packed)
        free_outputs[policy_name] = free
    kl = causal_kl(fp16_teacher['logits'], teacher['logits'])
    top1 = (fp16_teacher['logits'].argmax(-1) == teacher['logits'].argmax(-1)).float().mean().item()
    common_prefix = 0
    for reference, candidate in zip(baseline_tokens, free['tokens']):
        if reference != candidate:
            break
        common_prefix += 1
    report = packed.report()
    results.append({
        'policy': policy_name,
        'resident_bytes': report['resident_bytes'],
        'resident_mib': report['resident_bytes'] / 2**20,
        'compression_ratio': report['compression_ratio'],
        'teacher_mean_kl': float(kl.mean()),
        'teacher_max_kl': float(kl.max()),
        'teacher_top1_agreement': top1,
        'greedy_common_prefix': common_prefix,
        'greedy_exact_match': free['tokens'] == baseline_tokens,
        'decode_seconds': free['seconds'],
        'peak_gpu_gib_dense_reference': free['peak_gpu_gib'],
        'text': tokenizer.decode(free['tokens'], skip_special_tokens=True),
    })

results_table = pd.DataFrame(results)
display(results_table.drop(columns=['text']).round(6))
for row in results:
    print(f"\n[{row['policy']}] {row['text']!r}")

## Plot and export reproducible artifacts

A strong result is not necessarily 'INT4 wins.' Look for a Pareto tradeoff: declarative INT4 should use more bytes than uniform INT4 but may preserve FP16 behavior better by protecting the evidence. CUDA peak values describe this dense compatibility path, not a fused quantized kernel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(results_table.policy, results_table.resident_mib)
axes[0].set_ylabel('Physical packed cache (MiB)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].set_title('Storage')
axes[1].scatter(results_table.resident_mib, results_table.teacher_mean_kl, s=80)
for row in results:
    axes[1].annotate(row['policy'], (row['resident_mib'], row['teacher_mean_kl']))
axes[1].set_xlabel('Physical packed cache (MiB)')
axes[1].set_ylabel('Teacher-forced mean KL vs FP16')
axes[1].set_title('Storage–fidelity tradeoff')
fig.tight_layout()
plot_path = OUTPUT_DIR / 'storage-fidelity.png'
fig.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()

artifact = {
    'schema_version': 1,
    'model_id': MODEL_ID,
    'model_revision': revision,
    'gpu': GPU_NAME,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'seed': SEED,
    'prompt_tokens': int(input_ids.shape[1]),
    'max_new_tokens': MAX_NEW_TOKENS,
    'teacher_forced_steps': TEACHER_FORCED_STEPS,
    'group_size': GROUP_SIZE,
    'declarative_program': declarative_program,
    'segments': [vars(segment) for segment in segments],
    'results': results,
    'interpretation_note': (
        'resident_bytes counts physical packed payloads and scales. '
        'peak_gpu_gib_dense_reference includes a materialized FP16 cache because stock attention '
        'does not consume heterogeneous packed blocks directly.'
    ),
}
json_path = OUTPUT_DIR / 'requantization-results.json'
json_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
results_table.drop(columns=['text']).to_csv(OUTPUT_DIR / 'requantization-results.csv', index=False)
print('Saved:', json_path, plot_path)

In [ ]:
# Optional: download a zip containing JSON, CSV, and the figure.
import shutil
archive = shutil.make_archive('/content/anchorkv-requantization-results', 'zip', OUTPUT_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not running in Colab; use the path printed above.')